In [6]:
# Cell 1 — Imports and paths
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gc
from pathlib import Path
from scipy.sparse import issparse
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase3"
RESULTS_DIR = PROJECT_DIR / "results" / "phase3"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

cluster_labels_1 = {
    "0": "T cells (resting)", "1": "T cells (naive/memory)",
    "2": "NK/Cytotoxic T cells", "3": "Activated T cells",
    "4": "Macrophages", "5": "Monocytes/DC"
}

print("Ready")

Ready


In [7]:
# Cell 2 — Load raw counts and add cell type labels
adata1 = sc.read_h5ad(PROCESSED_DIR / "GSE114725_phase1_v2_rawcounts.h5ad")

labels1 = sc.read_h5ad(
    PROCESSED_DIR / "GSE114725_phase2_v2_annotated.h5ad"
).obs[["leiden_0.8"]].copy()

labels1["cell_type"] = labels1["leiden_0.8"].map(cluster_labels_1)
adata1.obs["cell_type"] = labels1["cell_type"].reindex(adata1.obs_names).values

gc.collect()

print(adata1)
print(adata1.obs["cell_type"].value_counts())
print("Tissue types:", adata1.obs["tissue"].unique().tolist())
print("Max value:", adata1.X.max())

AnnData object with n_obs × n_vars = 44662 × 14800
    obs: 'patient', 'tissue', 'replicate', 'cluster', 'n_genes_by_counts', 'total_counts', 'doublet_score', 'predicted_doublet', 'cell_type'
    var: 'n_cells'
    uns: 'log1p'
cell_type
T cells (resting)         11734
Macrophages                8691
NK/Cytotoxic T cells       8676
Activated T cells          7141
T cells (naive/memory)     4341
Monocytes/DC               4079
Name: count, dtype: int64
Tissue types: ['TUMOR', 'NORMAL', 'LYMPHNODE', 'BLOOD']
Max value: 8.170489


In [8]:
# Cell 3 — Pseudobulk aggregation function
def pseudobulk_aggregate(adata, cell_type, sample_col, condition_col,
                          cell_type_col="cell_type"):
    mask = (adata.obs[cell_type_col] == cell_type).values
    adata_ct = adata[mask]
    
    print(f"\n{cell_type}: {adata_ct.n_obs} cells")
    
    samples = adata_ct.obs[sample_col].unique()
    counts_list = []
    meta_list = []
    
    for sample in samples:
        sample_mask = (adata_ct.obs[sample_col] == sample).values
        X_sample = adata_ct.X[sample_mask]
        
        if issparse(X_sample):
            X_sample = X_sample.toarray()
        
        counts_list.append(X_sample.sum(axis=0))
        
        condition = adata_ct.obs.loc[
            adata_ct.obs[sample_col] == sample,
            condition_col
        ].iloc[0]
        
        meta_list.append({
            sample_col: sample,
            condition_col: condition
        })
    
    counts_df = pd.DataFrame(
        np.vstack(counts_list),
        index=[m[sample_col] for m in meta_list],
        columns=adata_ct.var_names
    ).astype(int)
    
    meta_df = pd.DataFrame(meta_list).set_index(sample_col)
    
    print(f"  Pseudobulk matrix: {counts_df.shape}")
    print(f"  Conditions: {meta_df[condition_col].value_counts().to_dict()}")
    
    return counts_df, meta_df

print("Function defined")

Function defined


In [9]:
# Cell 4 — Run pseudobulk DE: T cells (resting) TUMOR vs BLOOD
counts_df, meta_df = pseudobulk_aggregate(
    adata1,
    cell_type="T cells (resting)",
    sample_col="patient",
    condition_col="tissue"
)

# Filter to TUMOR vs BLOOD only
mask = meta_df["tissue"].isin(["TUMOR", "BLOOD"])
counts_df = counts_df[mask]
meta_df = meta_df[mask]

print("\nFiltered metadata:")
print(meta_df)
print("\nCounts matrix shape:", counts_df.shape)
print("Sample counts (first 5 genes):")
print(counts_df.iloc[:, :5])


T cells (resting): 11734 cells
  Pseudobulk matrix: (8, 14800)
  Conditions: {'TUMOR': 3, 'NORMAL': 2, 'BLOOD': 2, 'LYMPHNODE': 1}

Filtered metadata:
        tissue
patient       
BC5      TUMOR
BC6      TUMOR
BC4      BLOOD
BC8      TUMOR
BC1      BLOOD

Counts matrix shape: (5, 14800)
Sample counts (first 5 genes):
     A1BG  A2M  A4GALT  AAAS  AACS
BC5     9    2       2     8     4
BC6    43   45       0    12    10
BC4   509   45       2   305   154
BC8    72   55       0    39     8
BC1   293   15       0    63    47


In [11]:
# Cell 5 — Run PyDESeq2
inference = DefaultInference()

dds = DeseqDataSet(
    counts=counts_df,
    metadata=meta_df,
    design="~tissue",
    refit_cooks=True,
    inference=inference
)

dds.deseq2()

stat_res = DeseqStats(
    dds,
    contrast=["tissue", "TUMOR", "BLOOD"],
    inference=inference
)
stat_res.summary()

results_df = stat_res.results_df
print("\nDE results shape:", results_df.shape)
print(results_df.head(10))

Fitting size factors...
... done in 0.02 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 7.83 seconds.

Fitting dispersion trend curve...
... done in 1.21 seconds.

C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:541: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()
Fitting MAP dispersions...
... done in 7.12 seconds.

Fitting LFCs...
... done in 5.27 seconds.

Calculating cook's distance...
... done in 0.03 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 3.07 seconds.



Log2 fold change & Wald test p-value: tissue TUMOR vs BLOOD
          baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1BG    100.505627       -0.953306  0.545830 -1.746525  0.080720  0.453021
A2M      38.319755        2.742585  0.799644  3.429760  0.000604  0.015392
A4GALT    1.166872        2.949114  3.613279  0.816188  0.414393       NaN
AAAS     39.956549       -0.392882  0.624035 -0.629584  0.528967  0.910009
AACS     19.915768       -1.041107  0.762207 -1.365911  0.171967  0.644533
...            ...             ...       ...       ...       ...       ...
ZXDC     39.553216       -0.438461  0.598639 -0.732430  0.463906  0.883077
ZYG11B   52.172902       -0.550829  0.519488 -1.060330  0.288995  0.768913
ZYX     364.060762        0.001300  0.357036  0.003642  0.997094  0.999625
ZZEF1    89.007546       -0.211750  0.402319 -0.526323  0.598664  0.925099
ZZZ3     35.950273       -0.653884  0.581977 -1.123556  0.261201  0.741917

[14800 rows x 6 columns]

DE results sh

In [12]:
# Cell 6 — Filter significant results and save
sig_df = results_df[
    (results_df["padj"] < 0.05) &
    (abs(results_df["log2FoldChange"]) > 0.5)
].copy()

sig_df = sig_df.sort_values("padj")

print(f"Significant DEGs: {len(sig_df)}")
print("\nTop upregulated in TUMOR:")
print(sig_df[sig_df["log2FoldChange"] > 0].head(10)[["log2FoldChange", "padj"]])
print("\nTop downregulated in TUMOR:")
print(sig_df[sig_df["log2FoldChange"] < 0].head(10)[["log2FoldChange", "padj"]])

results_df.to_csv(
    RESULTS_DIR / "GSE114725_DE_Tcells_resting_tumor_vs_blood.csv"
)
sig_df.to_csv(
    RESULTS_DIR / "GSE114725_DE_Tcells_resting_tumor_vs_blood_significant.csv"
)
print("\nSaved")

Significant DEGs: 601

Top upregulated in TUMOR:
          log2FoldChange          padj
HLA-DRA         2.370183  1.301097e-24
ATF3            5.026530  3.558711e-24
LYZ             4.212372  5.649935e-22
CCL3            3.146662  4.076705e-19
DUSP4           3.847187  1.191198e-17
SGK1            3.209662  5.102780e-16
CD163           6.082647  1.108901e-15
HLA-DPA1        2.084260  1.108901e-15
DUSP2           1.988918  7.242165e-15
KLF4            3.992785  2.182498e-14

Top downregulated in TUMOR:
                log2FoldChange          padj
TIPIN                -3.769220  1.867508e-21
NOSIP                -2.562704  6.041420e-14
CTD-2192J16.22       -2.049773  1.308714e-13
RP11-255M2.3         -4.497167  4.607259e-10
LINC00861            -2.440483  4.157993e-09
TCF7                 -1.499511  2.343567e-08
NACA2                -2.010141  2.485961e-08
GLTSCR2              -1.272002  3.404337e-08
EEF1B2               -1.212180  1.053353e-07
RPL29                -1.330355  5.292750e-0

In [14]:
# Cell 7 — Loop DE across all cell types
cell_types_de = [
    "T cells (naive/memory)",
    "NK/Cytotoxic T cells",
    "Activated T cells",
    "Macrophages"
]

all_results = {}

for ct in cell_types_de:
    print(f"\n{'='*50}")
    print(f"Running DE for: {ct}")
    print('='*50)
    
    try:
        counts_df, meta_df = pseudobulk_aggregate(
            adata1,
            cell_type=ct,
            sample_col="patient",
            condition_col="tissue"
        )
        
        # Filter to TUMOR vs BLOOD
        mask = meta_df["tissue"].isin(["TUMOR", "BLOOD"])
        counts_df = counts_df[mask]
        meta_df = meta_df[mask]
        
        # Need at least 2 samples per condition
        condition_counts = meta_df["tissue"].value_counts()
        if condition_counts.min() < 2:
            print(f"  Skipping - not enough samples: {condition_counts.to_dict()}")
            continue
        
        # Run PyDESeq2
        inference = DefaultInference()
        dds = DeseqDataSet(
            counts=counts_df,
            metadata=meta_df,
            design="~tissue",
            refit_cooks=True,
            inference=inference
        )
        dds.deseq2()
        
        stat_res = DeseqStats(
            dds,
            contrast=["tissue", "TUMOR", "BLOOD"],
            inference=inference
        )
        stat_res.summary()
        results_df = stat_res.results_df
        
        # Filter significant
        sig_df = results_df[
            (results_df["padj"] < 0.05) &
            (abs(results_df["log2FoldChange"]) > 0.5)
        ].copy().sort_values("padj")
        
        print(f"  Significant DEGs: {len(sig_df)}")
        
        # Save
        ct_clean = ct.replace("/", "_").replace(" ", "_")
        results_df.to_csv(
            RESULTS_DIR / f"GSE114725_DE_{ct_clean}_tumor_vs_blood.csv"
        )
        sig_df.to_csv(
            RESULTS_DIR / f"GSE114725_DE_{ct_clean}_tumor_vs_blood_significant.csv"
        )
        
        all_results[ct] = {"full": results_df, "sig": sig_df}
        
    except Exception as e:
        print(f"  Failed: {e}")

print("\nAll DE analyses complete")


Running DE for: T cells (naive/memory)

T cells (naive/memory): 4341 cells
  Pseudobulk matrix: (8, 14800)
  Conditions: {'TUMOR': 3, 'NORMAL': 2, 'BLOOD': 2, 'LYMPHNODE': 1}


Fitting size factors...
... done in 0.01 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 7.22 seconds.

Fitting dispersion trend curve...
... done in 1.14 seconds.

C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:541: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()
Fitting MAP dispersions...
... done in 6.44 seconds.

Fitting LFCs...
... done in 5.38 seconds.

Calculating cook's distance...
... done in 0.02 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 3.11 seconds.



Log2 fold change & Wald test p-value: tissue TUMOR vs BLOOD
          baseMean  log2FoldChange     lfcSE      stat    pvalue    padj
A1BG     28.044609       -1.172274  0.681098 -1.721154  0.085223     NaN
A2M       6.646336        2.093911  1.388028  1.508551  0.131414     NaN
A4GALT    0.199973       -0.367400  4.675395 -0.078582  0.937365     NaN
AAAS      8.681757       -0.195319  0.995869 -0.196129  0.844509     NaN
AACS      6.589275       -1.854132  1.303401 -1.422534  0.154871     NaN
...            ...             ...       ...       ...       ...     ...
ZXDC      9.743484       -0.948449  1.232691 -0.769413  0.441648     NaN
ZYG11B   15.762889        0.046730  0.866388  0.053937  0.956986     NaN
ZYX     102.646024       -0.254668  0.530821 -0.479763  0.631396  0.9106
ZZEF1    27.237556       -0.675252  0.603449 -1.118989  0.263145     NaN
ZZZ3     12.914374       -0.786872  0.967945 -0.812930  0.416258     NaN

[14800 rows x 6 columns]
  Significant DEGs: 202

Running DE fo

...

Log2 fold change & Wald test p-value: tissue TUMOR vs BLOOD
          baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1BG     86.408201       -0.750495  0.523422 -1.433824  0.151622  0.745231
A2M      79.002431        0.185112  0.670598  0.276041  0.782517  0.992030
A4GALT    2.647931       -3.604479  2.561484 -1.407184  0.159373       NaN
AAAS     37.271207        0.189574  0.596551  0.317784  0.750649  0.991373
AACS     19.703979        0.172865  0.802294  0.215463  0.829406  0.993304
...            ...             ...       ...       ...       ...       ...
ZXDC     34.663375       -0.300173  0.693118 -0.433077  0.664959  0.988068
ZYG11B   49.556252       -0.085941  0.599526 -0.143349  0.886015  0.993304
ZYX     400.118863        0.141006  0.323827  0.435435  0.663247  0.988068
ZZEF1    92.445476       -0.217478  0.399879 -0.543860  0.586538  0.980608
ZZZ3     37.009228       -0.999472  0.603132 -1.657137  0.097492  0.643106

[14800 rows x 6 columns]
  Significant 

Fitting size factors...
... done in 0.01 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 12.29 seconds.

Fitting dispersion trend curve...
... done in 1.77 seconds.

C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:541: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()
Fitting MAP dispersions...
... done in 13.48 seconds.

Fitting LFCs...
... done in 8.44 seconds.

Calculating cook's distance...
... done in 0.04 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 3.94 seconds.



Log2 fold change & Wald test p-value: tissue TUMOR vs BLOOD
          baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1BG    109.034313       -1.001316  0.505816 -1.979604  0.047748  0.284723
A2M     304.658203        1.464778  1.401143  1.045416  0.295831  0.694602
A4GALT    3.809885        0.629523  2.284968  0.275506  0.782927       NaN
AAAS     50.421978       -0.217259  0.582478 -0.372991  0.709155  0.917662
AACS     26.083255        0.442139  0.813638  0.543410  0.586848  0.872952
...            ...             ...       ...       ...       ...       ...
ZXDC     51.213505       -0.241679  0.645750 -0.374262  0.708210  0.917412
ZYG11B   85.119188        0.293456  0.569757  0.515055  0.606515  0.881999
ZYX     616.538593        0.536644  0.368485  1.456354  0.145295  0.513422
ZZEF1   122.941934        0.207719  0.470969  0.441047  0.659179  0.900910
ZZZ3     58.664243       -0.542610  0.574551 -0.944407  0.344962  0.731527

[14800 rows x 6 columns]
  Significant 

In [15]:
# Cell 8 — Volcano plots for all cell types
def volcano_plot(results_csv, title, save_path, lfc_thresh=0.5, pval_thresh=0.05):
    de_df = pd.read_csv(results_csv, index_col=0)
    de_df = de_df.dropna(subset=["padj", "log2FoldChange"])
    de_df["-log10_pval"] = -np.log10(de_df["padj"].clip(lower=1e-300))
    
    de_df["colour"] = "grey"
    de_df.loc[
        (de_df["log2FoldChange"] > lfc_thresh) & (de_df["padj"] < pval_thresh),
        "colour"
    ] = "red"
    de_df.loc[
        (de_df["log2FoldChange"] < -lfc_thresh) & (de_df["padj"] < pval_thresh),
        "colour"
    ] = "blue"
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    for colour, group in de_df.groupby("colour"):
        ax.scatter(
            group["log2FoldChange"],
            group["-log10_pval"],
            c=colour,
            alpha=0.5,
            s=10,
            label=colour
        )
    
    ax.axvline(x=lfc_thresh, color="black", linestyle="--", linewidth=0.8)
    ax.axvline(x=-lfc_thresh, color="black", linestyle="--", linewidth=0.8)
    ax.axhline(y=-np.log10(pval_thresh), color="black", linestyle="--", linewidth=0.8)
    
    # Label top genes
    top_up = de_df[de_df["colour"] == "red"].nlargest(5, "-log10_pval")
    top_down = de_df[de_df["colour"] == "blue"].nlargest(5, "-log10_pval")
    
    for gene, row in pd.concat([top_up, top_down]).iterrows():
        ax.annotate(
            gene,
            (row["log2FoldChange"], row["-log10_pval"]),
            fontsize=7,
            ha="center",
            xytext=(0, 5),
            textcoords="offset points"
        )
    
    n_up = (de_df["colour"] == "red").sum()
    n_down = (de_df["colour"] == "blue").sum()
    
    ax.text(0.98, 0.98, f"Up: {n_up}\nDown: {n_down}",
            transform=ax.transAxes, ha="right", va="top", fontsize=9)
    
    ax.set_xlabel("Log2 Fold Change (Tumour vs Blood)")
    ax.set_ylabel("-log10 Adjusted P-value")
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"Saved: {save_path.name}")

# Plot for all cell types
cell_types_plot = {
    "T_cells__resting_": "T cells (resting)",
    "T_cells__naive_memory_": "T cells (naive/memory)",
    "NK_Cytotoxic_T_cells": "NK/Cytotoxic T cells",
    "Activated_T_cells": "Activated T cells",
    "Macrophages": "Macrophages"
}

for ct_file, ct_label in cell_types_plot.items():
    csv_path = RESULTS_DIR / f"GSE114725_DE_{ct_file}_tumor_vs_blood.csv"
    if csv_path.exists():
        volcano_plot(
            csv_path,
            title=f"{ct_label} — Tumour vs Blood (GSE114725)",
            save_path=FIGURE_DIR / f"GSE114725_volcano_{ct_file}.png"
        )
    else:
        print(f"File not found: {csv_path.name}")

File not found: GSE114725_DE_T_cells__resting__tumor_vs_blood.csv
File not found: GSE114725_DE_T_cells__naive_memory__tumor_vs_blood.csv
Saved: GSE114725_volcano_NK_Cytotoxic_T_cells.png
Saved: GSE114725_volcano_Activated_T_cells.png
Saved: GSE114725_volcano_Macrophages.png
